# Marketing Funnel & Conversion Performance Analysis
### Data Science & Analytics – Task 3 (2026) | Future Interns

**Dataset:** Bank Marketing Dataset  
**Objective:** Analyze how customers move through the marketing funnel from initial contact to conversion (term deposit subscription), identify drop-off points, and provide actionable recommendations.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

# ── Global style ───────────────────────────────────────────────────────────────
PALETTE   = ["#1a3a5c", "#2e7d9e", "#4caf8a", "#f4a641", "#e05c5c"]
BG_COLOR  = "#f8f9fa"
plt.rcParams.update({
    "figure.facecolor": BG_COLOR,
    "axes.facecolor":   BG_COLOR,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.titlesize":   14,
    "axes.titleweight": "bold",
    "axes.labelsize":   11,
})
print("Libraries loaded successfully!")

## 1. Data Loading & Exploration

In [ ]:
df = pd.read_csv("bank-additional-full.csv", sep=";")

print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Names: {list(df.columns)}")
print(f"\nTarget Distribution:")
print(df['y'].value_counts())
print(f"\nConversion Rate: {df[df['y']=='yes'].shape[0]/len(df)*100:.2f}%")
df.head()

In [ ]:
# Basic statistics
print("Missing Values:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)

In [ ]:
# Encode target variable
df["subscribed"] = (df["y"] == "yes").astype(int)
print("Target encoding done: 'yes'=1, 'no'=0")
print(df[["y","subscribed"]].value_counts())

## 2. Marketing Funnel Definition

We model the bank marketing campaign as a 5-stage funnel:

| Stage | Definition |
|---|---|
| **Leads Contacted** | All records — every prospect was called |
| **Call Connected** | Duration > 0 seconds |
| **Engaged** | Duration ≥ 180 seconds (≥3 minute conversation) |
| **Warm Leads** | Previous successful contact or prior interactions |
| **Subscribed** | Target `y = 'yes'` — converted to term deposit |

In [ ]:
total       = len(df)
reached     = len(df[df["duration"] > 0])
engaged     = len(df[df["duration"] >= 180])
warm_lead   = len(df[(df["duration"] >= 180) & (df["poutcome"] == "success") | (df["previous"] > 0)])
converted   = int(df["subscribed"].sum())

funnel_stages  = ["Leads\nContacted", "Call\nConnected", "Engaged\n(≥3 min)", "Warm\nLeads", "Subscribed"]
funnel_values  = [total, reached, engaged, warm_lead, converted]

funnel_df = pd.DataFrame({"Stage": funnel_stages, "Count": funnel_values})
funnel_df["Conv_from_prev"] = funnel_df["Count"] / funnel_df["Count"].shift(1) * 100
funnel_df["Conv_from_top"]  = funnel_df["Count"] / funnel_df["Count"].iloc[0] * 100
funnel_df["Dropoff_%"]      = 100 - funnel_df["Conv_from_prev"]

print("FUNNEL ANALYSIS")
print(funnel_df.to_string(index=False))

In [ ]:
# Funnel Chart
fig, ax = plt.subplots(figsize=(10, 7))
bar_colors = ["#1a3a5c", "#2e7d9e", "#4caf8a", "#f4a641", "#e05c5c"]
max_val = funnel_values[0]

for i, (stage, val, color) in enumerate(zip(funnel_stages, funnel_values, bar_colors)):
    width = val / max_val
    left  = (1 - width) / 2
    ax.barh(len(funnel_stages) - i - 1, width, left=left, color=color,
            height=0.6, edgecolor="white", linewidth=1.5)
    pct = val / max_val * 100
    clean_stage = stage.replace("\n", " ")
    ax.text(0.5, len(funnel_stages) - i - 1,
            f"{clean_stage}  |  {val:,}  ({pct:.1f}%)",
            ha="center", va="center", color="white", fontsize=11, fontweight="bold")

ax.set_xlim(0, 1)
ax.axis("off")
ax.set_title("Marketing Funnel – Conversion Performance", pad=20)
plt.tight_layout()
plt.savefig("outputs/01_funnel_chart.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Drop-off Analysis

In [ ]:
dropoff_stages = funnel_stages[1:]
dropoffs = [funnel_values[i-1] - funnel_values[i] for i in range(1, len(funnel_values))]
dropoff_pct = [d / funnel_values[0] * 100 for d in dropoffs]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(dropoff_stages)), dropoff_pct,
              color=[PALETTE[4], PALETTE[3], PALETTE[1], PALETTE[2]],
              edgecolor="white", width=0.55)

for bar, pct in zip(bars, dropoff_pct):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{pct:.1f}%", ha="center", va="bottom", fontweight="bold")

labels = ["Disconnected\nCalls", "Short\nEngagement", "Lost After\nEngagement", "Not\nConverted"]
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("Drop-off (% of total leads)")
ax.set_title("Funnel Drop-off Analysis by Stage")
plt.tight_layout()
plt.savefig("outputs/04_dropoff_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Conversion Rate by Segment

In [ ]:
# By Job
job_conv = (df.groupby("job")["subscribed"]
              .agg(["sum","count"])
              .rename(columns={"sum":"Converted","count":"Total"}))
job_conv["Rate"] = job_conv["Converted"] / job_conv["Total"] * 100
job_conv = job_conv.sort_values("Rate", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
avg = job_conv["Rate"].mean()
colors = [PALETTE[2] if r > avg else PALETTE[0] for r in job_conv["Rate"]]
bars = ax.barh(job_conv.index, job_conv["Rate"], color=colors, edgecolor="white")
ax.axvline(avg, color=PALETTE[3], linestyle="--", linewidth=1.5, label=f"Avg: {avg:.1f}%")
for bar in bars:
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.1f}%", va="center", fontsize=9)
ax.set_xlabel("Conversion Rate (%)")
ax.set_title("Conversion Rate by Job Category")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/02_conversion_by_job.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Monthly funnel
month_order = ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]
month_conv = (df.groupby("month")["subscribed"]
               .agg(["sum","count"])
               .rename(columns={"sum":"Converted","count":"Total"}))
month_conv["Rate"] = month_conv["Converted"] / month_conv["Total"] * 100
month_conv = month_conv.reindex([m for m in month_order if m in month_conv.index])

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.bar(month_conv.index, month_conv["Total"], color=PALETTE[0], alpha=0.5, label="Total Contacted")
ax2.plot(month_conv.index, month_conv["Rate"], color=PALETTE[4],
         marker="o", linewidth=2.5, markersize=7, label="Conversion Rate %")
ax2.fill_between(range(len(month_conv)), month_conv["Rate"], alpha=0.1, color=PALETTE[4])
ax1.set_ylabel("Number of Contacts", color=PALETTE[0])
ax2.set_ylabel("Conversion Rate (%)", color=PALETTE[4])
ax1.set_title("Monthly Contact Volume vs Conversion Rate")
ax1.set_xticks(range(len(month_conv)))
ax1.set_xticklabels([m.capitalize() for m in month_conv.index], rotation=45)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
plt.tight_layout()
plt.savefig("outputs/03_monthly_funnel.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Channel and Education
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

contact_conv = (df.groupby("contact")["subscribed"]
                  .agg(["sum","count"])
                  .rename(columns={"sum":"Conv","count":"Total"}))
contact_conv["Rate"] = contact_conv["Conv"] / contact_conv["Total"] * 100

edu_order = ["basic.4y","basic.6y","basic.9y","high.school","professional.course","university.degree"]
edu_conv = (df.groupby("education")["subscribed"]
              .agg(["sum","count"])
              .rename(columns={"sum":"Conv","count":"Total"}))
edu_conv["Rate"] = edu_conv["Conv"] / edu_conv["Total"] * 100
edu_conv = edu_conv.reindex([e for e in edu_order if e in edu_conv.index])

axes[0].bar(contact_conv.index, contact_conv["Rate"], color=PALETTE[:2], width=0.4, edgecolor="white")
for i, (idx, row) in enumerate(contact_conv.iterrows()):
    axes[0].text(i, row["Rate"] + 0.3, f"{row['Rate']:.1f}%", ha="center", fontweight="bold")
axes[0].set_title("Conversion Rate by Contact Channel")
axes[0].set_ylabel("Conversion Rate (%)")

axes[1].bar(range(len(edu_conv)), edu_conv["Rate"], color=PALETTE[2], edgecolor="white")
for i, (idx, row) in enumerate(edu_conv.iterrows()):
    axes[1].text(i, row["Rate"] + 0.2, f"{row['Rate']:.1f}%", ha="center", fontsize=9, fontweight="bold")
axes[1].set_xticks(range(len(edu_conv)))
axes[1].set_xticklabels([e.replace(".", " ").title() for e in edu_conv.index], rotation=30, ha="right")
axes[1].set_title("Conversion Rate by Education Level")
axes[1].set_ylabel("Conversion Rate (%)")
plt.suptitle("Channel & Education Impact on Conversion", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("outputs/05_channel_education.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Call Duration Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

yes_dur = df[df["y"] == "yes"]["duration"] / 60
no_dur  = df[df["y"] == "no"]["duration"] / 60

axes[0].boxplot([no_dur, yes_dur], labels=["Not Subscribed", "Subscribed"],
                patch_artist=True,
                boxprops=dict(facecolor=PALETTE[1], color=PALETTE[0]),
                medianprops=dict(color=PALETTE[4], linewidth=2.5))
axes[0].set_ylabel("Call Duration (minutes)")
axes[0].set_title("Call Duration by Outcome")

bins = [0, 60, 180, 300, 600, df["duration"].max()]
labels_b = ["<1 min", "1-3 min", "3-5 min", "5-10 min", ">10 min"]
df["dur_bucket"] = pd.cut(df["duration"], bins=bins, labels=labels_b)
dur_conv = (df.groupby("dur_bucket", observed=True)["subscribed"]
              .agg(["sum","count"])
              .rename(columns={"sum":"Conv","count":"Total"}))
dur_conv["Rate"] = dur_conv["Conv"] / dur_conv["Total"] * 100

axes[1].bar(range(len(dur_conv)), dur_conv["Rate"],
            color=[PALETTE[i % len(PALETTE)] for i in range(len(dur_conv))], edgecolor="white")
for i, (_, row) in enumerate(dur_conv.iterrows()):
    axes[1].text(i, row["Rate"] + 0.5, f"{row['Rate']:.1f}%", ha="center", fontsize=9, fontweight="bold")
axes[1].set_xticks(range(len(dur_conv)))
axes[1].set_xticklabels(labels_b)
axes[1].set_ylabel("Conversion Rate (%)")
axes[1].set_title("Conversion Rate by Call Duration Bucket")
plt.suptitle("Call Duration Impact on Conversion", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/06_duration_impact.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Campaign Frequency & Previous Outcome

In [ ]:
camp_conv = (df.groupby("campaign")["subscribed"]
               .agg(["sum","count"])
               .rename(columns={"sum":"Conv","count":"Total"}))
camp_conv["Rate"] = camp_conv["Conv"] / camp_conv["Total"] * 100
camp_conv = camp_conv[camp_conv["Total"] >= 50].head(15)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()
ax1.bar(camp_conv.index.astype(str), camp_conv["Total"],
        color=PALETTE[0], alpha=0.55, label="Total Contacted")
ax2.plot(camp_conv.index.astype(str), camp_conv["Rate"],
         color=PALETTE[4], marker="o", linewidth=2.5, markersize=7, label="Conversion Rate %")
ax1.set_xlabel("Number of Campaign Contacts")
ax1.set_ylabel("Number of Contacts", color=PALETTE[0])
ax2.set_ylabel("Conversion Rate (%)", color=PALETTE[4])
ax1.set_title("Campaign Contact Frequency vs Conversion Rate")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
plt.tight_layout()
plt.savefig("outputs/07_campaign_frequency.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
pout_conv = (df.groupby("poutcome")["subscribed"]
               .agg(["sum","count"])
               .rename(columns={"sum":"Conv","count":"Total"}))
pout_conv["Rate"] = pout_conv["Conv"] / pout_conv["Total"] * 100

bins_age   = [0, 25, 35, 45, 55, 65, 100]
labels_age = ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]
df["age_group"] = pd.cut(df["age"], bins=bins_age, labels=labels_age)
age_conv = (df.groupby("age_group", observed=True)["subscribed"]
              .agg(["sum","count"])
              .rename(columns={"sum":"Conv","count":"Total"}))
age_conv["Rate"] = age_conv["Conv"] / age_conv["Total"] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_pout = [PALETTE[2] if r > 20 else (PALETTE[3] if r > 10 else PALETTE[4])
               for r in pout_conv["Rate"]]
axes[0].bar(pout_conv.index, pout_conv["Rate"], color=colors_pout, edgecolor="white", width=0.5)
for i, (idx, row) in enumerate(pout_conv.iterrows()):
    axes[0].text(i, row["Rate"] + 0.5, f"{row['Rate']:.1f}%", ha="center", fontweight="bold")
axes[0].set_title("Conversion Rate by Previous Outcome")
axes[0].set_ylabel("Conversion Rate (%)")

colors_age = [PALETTE[i % len(PALETTE)] for i in range(len(age_conv))]
axes[1].bar(range(len(age_conv)), age_conv["Rate"], color=colors_age, edgecolor="white")
for i, (_, row) in enumerate(age_conv.iterrows()):
    axes[1].text(i, row["Rate"] + 0.3, f"{row['Rate']:.1f}%", ha="center", fontweight="bold")
axes[1].set_xticks(range(len(age_conv)))
axes[1].set_xticklabels(labels_age)
axes[1].set_ylabel("Conversion Rate (%)")
axes[1].set_title("Conversion Rate by Age Group")
plt.suptitle("Previous Outcome & Age Impact on Conversion", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/08_poutcome_age.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. KPI Summary Dashboard

In [ ]:
overall_conv_rate = df["subscribed"].mean() * 100
avg_call_dur_yes  = df[df["y"] == "yes"]["duration"].mean() / 60
avg_call_dur_no   = df[df["y"] == "no"]["duration"].mean() / 60
best_month        = month_conv["Rate"].idxmax().capitalize()
best_job          = job_conv["Rate"].idxmax()
cellular_rate     = contact_conv.loc["cellular", "Rate"] if "cellular" in contact_conv.index else 0

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.patch.set_facecolor("#1a3a5c")
kpi_items = [
    ("Overall Conversion Rate", f"{overall_conv_rate:.1f}%",   "#4caf8a", f"of all {len(df):,} contacts subscribed"),
    ("Total Subscribers",       f"{converted:,}",               "#2e7d9e", f"out of {total:,} contacts"),
    ("Avg Call – Converted",    f"{avg_call_dur_yes:.1f} min",  "#f4a641", f"vs {avg_call_dur_no:.1f} min for non-converted"),
    ("Best Month",              best_month,                     "#e05c5c", "highest conversion rate"),
    ("Top Job Category",        best_job.title(),               "#8e44ad", "highest conversion rate"),
    ("Cellular Conv Rate",      f"{cellular_rate:.1f}%",        "#16a085", "vs telephone channel"),
]

for ax, (title, value, color, subtitle) in zip(axes.flat, kpi_items):
    ax.set_facecolor(color)
    ax.text(0.5, 0.6,  value,    ha="center", va="center", fontsize=26,
            fontweight="bold", color="white", transform=ax.transAxes)
    ax.text(0.5, 0.25, title,    ha="center", va="center", fontsize=11,
            color="white", transform=ax.transAxes, fontweight="bold")
    ax.text(0.5, 0.08, subtitle, ha="center", va="center", fontsize=8,
            color="white", alpha=0.85, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

fig.suptitle("Marketing Funnel KPI Dashboard", color="white", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("outputs/09_kpi_dashboard.png", dpi=150, bbox_inches="tight", facecolor="#1a3a5c")
plt.show()

## 8. Recommendations

Based on the funnel analysis, here are the top **7 actionable recommendations**:

| # | Recommendation | Expected Impact |
|---|---|---|
| 1 | **Focus on cellular channel** — 14.7% vs ~6.5% telephone conversion | High |
| 2 | **Train agents to keep calls past 3 minutes** — 4x higher conversion | High |
| 3 | **Prioritize warm leads** (prior successful outcome converts at 65%+) | High |
| 4 | **Cap contacts at 3 per prospect** — sharp drop-off after 3 calls | Medium |
| 5 | **Schedule campaigns in March, September, October** — best months | Medium |
| 6 | **Target student and retired segments** with tailored offers | Medium |
| 7 | **Build re-engagement pipelines** for prospects who engaged but didn't convert | Medium |

---

## Conclusion

The bank's marketing campaign achieves an **11.27% overall conversion rate** across 41,188 contacts. The biggest opportunity lies in:
- **Improving engagement quality** (call duration is the single strongest predictor)
- **Channel optimization** (cellular dramatically outperforms telephone)
- **Leveraging past success** (prior successful contacts are 6x more likely to convert)

By implementing these recommendations, the bank can realistically improve conversion rates to **15-20%** without increasing the number of contacts made.
